<a href="https://colab.research.google.com/github/gandirakesh/careerai-fullyworking/blob/main/query_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
from google.colab import files

uploaded = files.upload()

Saving patient_queries_5000.csv to patient_queries_5000 (1).csv


In [9]:
!pip install pandas scikit-learn joblib

In [10]:

import json
from pathlib import Path
import joblib
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# 1. Google Colab working directory
ROOT = Path("/content")

# 2. Load dataset
DATA_PATH = ROOT / "patient_queries_5000.csv"

df = pd.read_csv(DATA_PATH)

# Remove missing values
df = df.dropna(subset=["query", "category"])

df["query"] = df["query"].astype(str)
df["category"] = df["category"].astype(str)

print("Dataset loaded successfully!")
print("Total records:", len(df))
print("Total categories:", df["category"].nunique())

# 3. Split dataset
X_train, X_test, y_train, y_test = train_test_split(
    df["query"],
    df["category"],
    test_size=0.25,
    random_state=42,
    stratify=df["category"]
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# 4. Create NLP model
model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        strip_accents="unicode",
        ngram_range=(1, 2),
        max_features=8000,
        sublinear_tf=True,
        min_df=1
    )),
    ("classifier", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        C=2.0,
        random_state=42
    ))
])

# 5. Train model
print("\nTraining NLP model...")

model.fit(X_train, y_train)

# 6. Predictions
pred = model.predict(X_test)

# 7. Evaluation
report = classification_report(
    y_test,
    pred,
    output_dict=True,
    zero_division=0
)

metrics = {
    "accuracy": round(float(accuracy_score(y_test, pred)), 4),
    "macro_precision": round(report["macro avg"]["precision"], 4),
    "macro_recall": round(report["macro avg"]["recall"], 4),
    "macro_f1": round(report["macro avg"]["f1-score"], 4),
    "weighted_f1": round(report["weighted avg"]["f1-score"], 4),
    "class_report": report,
    "confusion_matrix": confusion_matrix(
        y_test, pred, labels=model.classes_
    ).tolist(),
    "labels": list(model.classes_),
    "training_samples": len(X_train),
    "testing_samples": len(X_test)
}

# 8. Save model
MODEL_DIR = ROOT / "models"
MODEL_DIR.mkdir(exist_ok=True)

joblib.dump(
    model,
    MODEL_DIR / "query_classifier.joblib"
)

with open(MODEL_DIR / "evaluation_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

# 9. Display results
print("\n================================")
print("NLP MODEL TRAINING COMPLETED")
print("================================")

print(f"Accuracy: {metrics['accuracy']:.2%}")
print(f"Macro Precision: {metrics['macro_precision']:.2%}")
print(f"Macro Recall: {metrics['macro_recall']:.2%}")
print(f"Macro F1: {metrics['macro_f1']:.2%}")
print(f"Weighted F1: {metrics['weighted_f1']:.2%}")

print("\nDetailed Classification Report:")
print(classification_report(y_test, pred, zero_division=0))

print("\nSaved model:", MODEL_DIR / "query_classifier.joblib")
print("Saved metrics:", MODEL_DIR / "evaluation_metrics.json")


Dataset loaded successfully!
Total records: 5000
Total categories: 13

Training samples: 3750
Testing samples: 1250

Training NLP model...

NLP MODEL TRAINING COMPLETED
Accuracy: 98.32%
Macro Precision: 98.35%
Macro Recall: 98.32%
Macro F1: 98.32%
Weighted F1: 98.32%

Detailed Classification Report:
                             precision    recall  f1-score   support

        Appointment Booking       1.00      0.98      0.99        96
   Appointment Cancellation       0.98      0.98      0.98        96
   Appointment Rescheduling       0.97      0.96      0.96        96
         Appointment Status       0.98      0.96      0.97        97
        Billing and Payment       0.94      0.99      0.96        95
     Department Information       0.98      1.00      0.99        96
         Doctor Information       0.99      0.95      0.97        96
            General Enquiry       0.99      0.99      0.99        97
           Hospital Timings       0.96      0.99      0.97        97
        

In [11]:
test_queries = [
    "I want to book an appointment with a doctor",
    "How can I cancel my appointment?",
    "Show me the available doctors",
    "I want to know my hospital bill",
    "What are the hospital opening hours?",
    "I need to download my blood test report",
    "Can I change my appointment date?"
]

for query in test_queries:
    prediction = model.predict([query])[0]
    print(f"Query: {query}")
    print(f"Predicted Category: {prediction}")
    print("-" * 50)

Query: I want to book an appointment with a doctor
Predicted Category: Appointment Booking
--------------------------------------------------
Query: How can I cancel my appointment?
Predicted Category: Appointment Cancellation
--------------------------------------------------
Query: Show me the available doctors
Predicted Category: Doctor Information
--------------------------------------------------
Query: I want to know my hospital bill
Predicted Category: Billing and Payment
--------------------------------------------------
Query: What are the hospital opening hours?
Predicted Category: Hospital Timings
--------------------------------------------------
Query: I need to download my blood test report
Predicted Category: Medical Report Query
--------------------------------------------------
Query: Can I change my appointment date?
Predicted Category: Appointment Rescheduling
--------------------------------------------------


In [12]:
import shutil

shutil.make_archive(
    "/content/hospital_nlp_models",
    "zip",
    "/content/models"
)

from google.colab import files
files.download("/content/hospital_nlp_models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>